# LumenY â€” Two-Stage Model: Volatility Filter + Direction Prediction

**Strategy:** Don't predict every hour blindly. First predict WHEN a big move is coming, then predict WHERE it goes.

**Stage 1 â€” Volatility Model:** LightGBM regression predicting `|log_return_1H|`. Trained on all data. Identifies hours where a meaningful move is expected.

**Stage 2 â€” Direction Model:** LightGBM binary classifier predicting UP/DOWN. Trained ONLY on rows where `|return| > median |return|` â€” the model only learns from hours where something real happened.

**Inference:** Vol model flags high-vol hours â†’ Direction model predicts sign â†’ Trade.

**Validation:** Walk-forward on training set. Final models trained up to TRAIN_END. Test set is genuinely unseen.

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import matplotlib.pyplot as plt
import joblib
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
from sklearn.metrics import mean_absolute_error, accuracy_score, classification_report

FEATURES_DIR = Path('../backend/data/features')
MODELS_DIR   = Path('../backend/models_4')
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# â”€â”€ CRITICAL: Training cutoff â”€â”€
TRAIN_END = '2024-06-30'

# Vol threshold: top X% of hours are "high vol"
VOL_PERCENTILE = 50  # median split â€” top 50% = high vol

print('Ready.')
print(f'Training cutoff: {TRAIN_END}')
print(f'Vol filter: top {100 - VOL_PERCENTILE}% of hours by |return|')

## 1. Load Dataset

In [ ]:
df = pd.read_parquet(FEATURES_DIR / 'all_pairs_features_labels.parquet')

label_cols   = [c for c in df.columns if c.startswith('label_')]
drop_cols    = label_cols + ['pair']
feature_cols = [c for c in df.columns if c not in drop_cols]

# Target for this notebook: 1H log return
TARGET_COL = 'label_1H'
df['abs_return'] = df[TARGET_COL].abs()
df['direction']  = (df[TARGET_COL] > 0).astype(int)  # 1=UP, 0=DOWN

# Split
df_train = df[df.index <= TRAIN_END].copy()
df_test  = df[df.index > TRAIN_END].copy()

print(f'Full dataset:  {df.shape}  ({df.index.min().date()} â†’ {df.index.max().date()})')
print(f'Features:      {len(feature_cols)}')
print(f'Train set:     {len(df_train):,} rows  ({df_train.index.min().date()} â†’ {df_train.index.max().date()})')
print(f'Test set:      {len(df_test):,} rows  ({df_test.index.min().date()} â†’ {df_test.index.max().date()})')
print(f'Pairs:         {df["pair"].unique()}')
print(f'\n|return| stats (train):')
print(df_train['abs_return'].describe())
print(f'\nMedian |return|: {df_train["abs_return"].median():.6f}')
print(f'Direction balance: {df_train["direction"].mean():.3f} UP')

## 2. Walk-Forward Splits

In [ ]:
def walk_forward_splits(n, n_splits=5, test_ratio=0.1):
    """Expanding window walk-forward splits. All within training data."""
    test_size = int(n * test_ratio)
    splits = []
    for i in range(n_splits):
        test_start = int(n * 0.5) + i * (int(n * 0.5) // n_splits)
        test_end   = test_start + test_size
        if test_end > n:
            break
        splits.append((list(range(0, test_start)), list(range(test_start, test_end))))
    return splits

splits = walk_forward_splits(len(df_train))
print(f'Walk-forward splits: {len(splits)} (all within training set, before {TRAIN_END})')
for i, (train_idx, test_idx) in enumerate(splits):
    print(f'  Fold {i+1}: Train â†’ {df_train.index[train_idx[-1]].date()} ({len(train_idx):,}) | '
          f'Test {df_train.index[test_idx[0]].date()} â†’ {df_train.index[test_idx[-1]].date()} ({len(test_idx):,})')

## 3. Stage 1 â€” Volatility Model

LightGBM regression predicting `|log_return_1H|`. This model learns WHEN big moves happen â€” volatility clustering, session effects, macro patterns.

In [ ]:
VOL_PARAMS = {
    'objective':         'regression',
    'metric':            'mae',
    'boosting_type':     'gbdt',
    'n_estimators':      5000,
    'learning_rate':     0.02,
    'num_leaves':        64,
    'max_depth':         6,
    'min_child_samples': 50,
    'feature_fraction':  0.7,
    'bagging_fraction':  0.8,
    'bagging_freq':      5,
    'reg_alpha':         0.1,
    'reg_lambda':        0.1,
    'random_state':      42,
    'n_jobs':            -1,
    'verbose':           -1,
    'device':            'gpu',
}

# Prepare data — predict |return|
X_train_all = df_train[feature_cols].ffill().fillna(0)
y_vol_all   = df_train['abs_return']
valid_mask  = y_vol_all.notna()
X_vol       = X_train_all[valid_mask]
y_vol       = y_vol_all[valid_mask]

print(f'Stage 1 — Volatility model')
print(f'Target: |log_return_1H|')
print(f'Samples: {len(X_vol):,}')
print(f'Target mean: {y_vol.mean():.6f}, median: {y_vol.median():.6f}')

# Walk-forward CV
vol_splits = walk_forward_splits(len(X_vol))
oof_vol_preds = np.full(len(X_vol), np.nan)
vol_best_iters = []
vol_fold_mae = []

for fold, (tr_idx, te_idx) in enumerate(vol_splits):
    X_tr, y_tr = X_vol.iloc[tr_idx], y_vol.iloc[tr_idx]
    X_te, y_te = X_vol.iloc[te_idx], y_vol.iloc[te_idx]
    
    model = lgb.LGBMRegressor(**VOL_PARAMS)
    model.fit(X_tr, y_tr, eval_set=[(X_te, y_te)],
              callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)])
    
    preds = model.predict(X_te)
    oof_vol_preds[te_idx] = preds
    mae = mean_absolute_error(y_te, preds)
    vol_fold_mae.append(mae)
    vol_best_iters.append(model.best_iteration_)
    print(f'  Fold {fold+1}: MAE={mae:.6f} | best_iter={model.best_iteration_}')

print(f'\nCV MAE: {np.mean(vol_fold_mae):.6f} ± {np.std(vol_fold_mae):.6f}')
print(f'Best iters: {vol_best_iters}')

# —— Key question: can the vol model distinguish high-vol from low-vol hours? ——
valid_oof = ~np.isnan(oof_vol_preds)
y_oof = y_vol.values[valid_oof]
p_oof = oof_vol_preds[valid_oof]

# Split predictions into quintiles and check actual vol in each
pred_bins = pd.qcut(p_oof, 5, labels=['Q1-Low', 'Q2', 'Q3', 'Q4', 'Q5-High'])
vol_by_bin = pd.DataFrame({'pred_bin': pred_bins, 'actual_vol': y_oof}).groupby('pred_bin')['actual_vol']
print(f'\nActual |return| by predicted vol quintile:')
print(vol_by_bin.mean().to_string())
print(f'\nRatio Q5/Q1: {vol_by_bin.mean().iloc[-1] / vol_by_bin.mean().iloc[0]:.2f}x')
print('(Higher = better vol discrimination)')

In [ ]:
# Train final vol model on ALL training data
avg_vol_iter = max(50, int(np.mean(vol_best_iters)))
print(f'Training final vol model with {avg_vol_iter} iterations on full training set...')

final_vol_model = lgb.LGBMRegressor(**{**VOL_PARAMS, 'n_estimators': avg_vol_iter})
final_vol_model.fit(X_vol, y_vol)

# Compute vol prediction distribution on TRAINING data (for leak-free thresholds)
vol_preds_train = final_vol_model.predict(X_vol)
vol_percentiles_train = {
    p: np.percentile(vol_preds_train, p) for p in [25, 50, 67, 80, 90]
}
print(f'\nVol prediction percentiles (from training set — used at inference):')
for p, v in vol_percentiles_train.items():
    print(f'  P{p}: {v:.6f}')

joblib.dump({
    'model': final_vol_model,
    'type': 'volatility',
    'target': '|log_return_1H|',
    'feature_cols': feature_cols,
    'train_end': TRAIN_END,
    'vol_percentiles': vol_percentiles_train,
}, MODELS_DIR / 'vol_model.joblib')

print(f'\nSaved: {MODELS_DIR / "vol_model.joblib"}')
print(f'Model size: {(MODELS_DIR / "vol_model.joblib").stat().st_size / 1024 / 1024:.1f} MB')

## 4. Stage 2 — Direction Model

LightGBM binary classifier predicting UP/DOWN. **Trained ONLY on high-vol rows** (above median |return|). The idea: when nothing moves, direction is noise. When something moves, there's signal.

In [ ]:
DIR_PARAMS = {
    'objective':         'binary',
    'metric':            'binary_logloss',
    'boosting_type':     'gbdt',
    'n_estimators':      5000,
    'learning_rate':     0.02,
    'num_leaves':        64,
    'max_depth':         6,
    'min_child_samples': 50,
    'feature_fraction':  0.7,
    'bagging_fraction':  0.8,
    'bagging_freq':      5,
    'reg_alpha':         0.1,
    'reg_lambda':        0.1,
    'random_state':      42,
    'n_jobs':            -1,
    'verbose':           -1,
    'device':            'gpu',
}

# Filter to high-vol rows only (above median |return|)
y_dir_all = df_train['direction']
valid_mask_dir = df_train[TARGET_COL].notna()
vol_threshold = df_train.loc[valid_mask_dir, 'abs_return'].median()

high_vol_mask = valid_mask_dir & (df_train['abs_return'] > vol_threshold)
X_dir = X_train_all[high_vol_mask]
y_dir = y_dir_all[high_vol_mask]

print(f'Stage 2 — Direction model')
print(f'Target: UP(1) / DOWN(0)')
print(f'Vol threshold (median): {vol_threshold:.6f}')
print(f'High-vol samples: {len(X_dir):,} ({len(X_dir)/valid_mask_dir.sum()*100:.1f}% of valid)')
print(f'Direction balance: {y_dir.mean():.3f} UP')

# Walk-forward CV on high-vol subset
dir_splits = walk_forward_splits(len(X_dir))
oof_dir_preds = np.full(len(X_dir), np.nan)
dir_best_iters = []
dir_fold_acc = []

for fold, (tr_idx, te_idx) in enumerate(dir_splits):
    X_tr, y_tr = X_dir.iloc[tr_idx], y_dir.iloc[tr_idx]
    X_te, y_te = X_dir.iloc[te_idx], y_dir.iloc[te_idx]
    
    model = lgb.LGBMClassifier(**DIR_PARAMS)
    model.fit(X_tr, y_tr, eval_set=[(X_te, y_te)],
              callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)])
    
    probs = model.predict_proba(X_te)[:, 1]
    preds = (probs > 0.5).astype(int)
    oof_dir_preds[te_idx] = probs
    
    acc = accuracy_score(y_te, preds)
    dir_fold_acc.append(acc)
    dir_best_iters.append(model.best_iteration_)
    print(f'  Fold {fold+1}: Accuracy={acc:.4f} ({acc*100:.1f}%) | best_iter={model.best_iteration_}')

print(f'\nCV Accuracy: {np.mean(dir_fold_acc):.4f} ({np.mean(dir_fold_acc)*100:.1f}%) ± {np.std(dir_fold_acc):.4f}')
print(f'Best iters: {dir_best_iters}')

# Accuracy by confidence bucket
valid_oof_dir = ~np.isnan(oof_dir_preds)
y_oof_dir = y_dir.values[valid_oof_dir]
p_oof_dir = oof_dir_preds[valid_oof_dir]

conf = np.abs(p_oof_dir - 0.5)
conf_bins = pd.qcut(conf, 5, labels=['Low', 'Q2', 'Q3', 'Q4', 'High'], duplicates='drop')
pred_labels = (p_oof_dir > 0.5).astype(int)
acc_by_conf = pd.DataFrame({'conf_bin': conf_bins, 'correct': (pred_labels == y_oof_dir)}).groupby('conf_bin')['correct']
print(f'\nAccuracy by confidence quintile:')
for name, grp in acc_by_conf:
    print(f'  {name}: {grp.mean():.3f} ({grp.mean()*100:.1f}%) — {len(grp):,} samples')

In [ ]:
# Train final direction model on ALL high-vol training data
avg_dir_iter = max(50, int(np.mean(dir_best_iters)))
print(f'Training final direction model with {avg_dir_iter} iterations on full high-vol training set...')

final_dir_model = lgb.LGBMClassifier(**{**DIR_PARAMS, 'n_estimators': avg_dir_iter})
final_dir_model.fit(X_dir, y_dir)

joblib.dump({
    'model': final_dir_model,
    'type': 'direction',
    'target': 'UP/DOWN on high-vol hours',
    'vol_threshold': vol_threshold,
    'feature_cols': feature_cols,
    'train_end': TRAIN_END,
}, MODELS_DIR / 'dir_model.joblib')

print(f'Saved: {MODELS_DIR / "dir_model.joblib"}')
print(f'Model size: {(MODELS_DIR / "dir_model.joblib").stat().st_size / 1024 / 1024:.1f} MB')

## 5. Combined Evaluation on UNSEEN Test Set

The moment of truth. Vol model filters â†’ Direction model predicts â†’ We measure PnL.

**Strategy:** Every hour, vol model predicts expected |move|. If predicted vol > threshold â†’ direction model predicts UP/DOWN â†’ trade. If below â†’ skip (flat).

In [ ]:
# ── Load final models ──
vol_bundle = joblib.load(MODELS_DIR / 'vol_model.joblib')
dir_bundle = joblib.load(MODELS_DIR / 'dir_model.joblib')
vol_model = vol_bundle['model']
dir_model = dir_bundle['model']
vol_pct = vol_bundle['vol_percentiles']  # thresholds from TRAINING data

# ── Prepare test data ──
X_test = df_test[feature_cols].ffill().fillna(0)
y_test_return = df_test[TARGET_COL]
valid_test = y_test_return.notna()
X_test_clean = X_test[valid_test]
y_test_clean = y_test_return[valid_test]

# ── Stage 1: Vol predictions on test ──
vol_preds_test = vol_model.predict(X_test_clean)

# ── Stage 2: Direction predictions on test (all rows, but we'll filter) ──
dir_probs_test = dir_model.predict_proba(X_test_clean)[:, 1]  # P(UP)

# ── Build results DataFrame ──
results = pd.DataFrame({
    'actual_return': y_test_clean.values,
    'pred_vol': vol_preds_test,
    'pred_p_up': dir_probs_test,
    'pred_dir': np.where(dir_probs_test > 0.5, 1, -1),  # +1=UP, -1=DOWN
    'actual_dir': np.where(y_test_clean.values > 0, 1, -1),
    'pair': df_test.loc[valid_test, 'pair'].values,
}, index=y_test_clean.index)

print(f'Test set: {len(results):,} rows ({results.index.min().date()} → {results.index.max().date()})')
print(f'Vol predictions — mean: {vol_preds_test.mean():.6f}, median: {np.median(vol_preds_test):.6f}')
print(f'Direction probs — mean: {dir_probs_test.mean():.3f} (0.5=balanced)')

# ── Evaluate at multiple vol thresholds (from TRAINING set — no leakage) ──
vol_thresholds = {
    'All hours (no filter)': 0,
    'Top 75%': vol_pct[25],
    'Top 50% (median)': vol_pct[50],
    'Top 33%': vol_pct[67],
    'Top 20%': vol_pct[80],
    'Top 10%': vol_pct[90],
}

print(f'\nVol thresholds from TRAINING set (no look-ahead):')
for name, thresh in vol_thresholds.items():
    print(f'  {name}: {thresh:.6f}')

print(f'\n{"Filter":<25} {"Trades":>8} {"Accuracy":>10} {"Avg PnL":>12} {"Total PnL":>12} {"Sharpe":>8}')
print('-' * 80)

for name, thresh in vol_thresholds.items():
    mask = results['pred_vol'] > thresh
    if mask.sum() == 0:
        continue
    subset = results[mask]
    
    # PnL = predicted direction × actual return
    pnl = subset['pred_dir'] * subset['actual_return']
    correct = (subset['pred_dir'] == subset['actual_dir']).mean()
    avg_pnl = pnl.mean()
    total_pnl = pnl.sum()
    sharpe = (pnl.mean() / pnl.std()) * np.sqrt(252 * 24) if pnl.std() > 0 else 0
    
    print(f'{name:<25} {mask.sum():>8,} {correct:>9.1%} {avg_pnl:>12.6f} {total_pnl:>12.4f} {sharpe:>8.2f}')

In [ ]:
# ── Per-pair breakdown (using training median vol threshold) ──
filtered = results[results['pred_vol'] > vol_pct[50]]

print(f'Per-pair results (vol filter = training P50 = {vol_pct[50]:.6f}):')
print(f'\n{"Pair":<10} {"Trades":>8} {"Accuracy":>10} {"Avg PnL":>12} {"Total PnL":>12} {"Sharpe":>8}')
print('-' * 65)

for pair in sorted(filtered['pair'].unique()):
    p = filtered[filtered['pair'] == pair]
    pnl = p['pred_dir'] * p['actual_return']
    correct = (p['pred_dir'] == p['actual_dir']).mean()
    sharpe = (pnl.mean() / pnl.std()) * np.sqrt(252 * 24) if pnl.std() > 0 else 0
    print(f'{pair:<10} {len(p):>8,} {correct:>9.1%} {pnl.mean():>12.6f} {pnl.sum():>12.4f} {sharpe:>8.2f}')

## 6. Cumulative PnL â€” Equity Curve

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.patch.set_facecolor('#080c14')

# Use training-derived thresholds (no look-ahead)
strategies = {
    'All hours (no filter)': results,
    f'Top 50% vol (>{vol_pct[50]:.5f})': results[results['pred_vol'] > vol_pct[50]],
    f'Top 33% vol (>{vol_pct[67]:.5f})': results[results['pred_vol'] > vol_pct[67]],
    f'Top 20% vol (>{vol_pct[80]:.5f})': results[results['pred_vol'] > vol_pct[80]],
}

for ax, (name, subset) in zip(axes.flatten(), strategies.items()):
    ax.set_facecolor('#080c14')
    
    pnl = subset['pred_dir'] * subset['actual_return']
    cum_pnl = pnl.cumsum()
    
    # Also plot per-pair
    for pair in sorted(subset['pair'].unique()):
        p = subset[subset['pair'] == pair]
        pair_pnl = (p['pred_dir'] * p['actual_return']).cumsum()
        ax.plot(pair_pnl.index, pair_pnl.values, alpha=0.3, linewidth=0.7)
    
    # Aggregate
    ax.plot(cum_pnl.index, cum_pnl.values, color='#4fc3f7', linewidth=2, label='All pairs')
    ax.axhline(0, color=(1,1,1,0.2), linewidth=1, linestyle='--')
    
    acc = (subset['pred_dir'] == subset['actual_dir']).mean()
    sharpe = (pnl.mean() / pnl.std()) * np.sqrt(252 * 24) if pnl.std() > 0 else 0
    
    ax.set_title(f'{name} — Acc: {acc:.1%}, Sharpe: {sharpe:.2f}', color='white', fontsize=10)
    ax.tick_params(colors='white')
    ax.set_ylabel('Cumulative log return', color='white', fontsize=8)
    for spine in ax.spines.values(): spine.set_edgecolor('#1a2332')

plt.suptitle('Two-Stage Model — Equity Curves on UNSEEN Test Set\n(Vol thresholds from training set)', color='white', fontsize=13)
plt.tight_layout()
plt.show()

## 7. Feature Importance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 10))
fig.patch.set_facecolor('#080c14')

for ax, (name, bundle_path) in zip(axes, [('Volatility Model', 'vol_model.joblib'), ('Direction Model', 'dir_model.joblib')]):
    bundle = joblib.load(MODELS_DIR / bundle_path)
    importance = pd.Series(bundle['model'].feature_importances_, index=feature_cols)
    importance = importance.sort_values(ascending=True).tail(25)
    
    ax.barh(importance.index, importance.values, color='#4fc3f7', alpha=0.8)
    ax.set_facecolor('#080c14')
    ax.tick_params(colors='white', labelsize=7)
    ax.set_title(f'{name} â€” Top 25 Features', color='white', fontsize=11)
    for spine in ax.spines.values(): spine.set_edgecolor('#1a2332')

plt.suptitle('Feature Importance â€” Two-Stage Model', color='white', fontsize=13)
plt.tight_layout()
plt.show()

## 8. Direction Accuracy vs Confidence + Vol Filter Heatmap

In [ ]:
# Heatmap: accuracy as function of vol threshold × direction confidence
# Vol thresholds from TRAINING set (no look-ahead)
vol_thresh_map = {0: 0, 25: vol_pct[25], 50: vol_pct[50], 67: vol_pct[67], 80: vol_pct[80], 90: vol_pct[90]}
vol_pcts = [0, 25, 50, 67, 80, 90]
conf_bins_edges = [0.50, 0.52, 0.54, 0.56, 0.58, 0.60, 1.0]

results['confidence'] = results['pred_p_up'].clip(0, 1)
results['confidence'] = results['confidence'].where(results['pred_dir'] == 1, 1 - results['confidence'])

heatmap = np.zeros((len(vol_pcts), len(conf_bins_edges) - 1))
counts  = np.zeros_like(heatmap)

for i, vp in enumerate(vol_pcts):
    vthresh = vol_thresh_map[vp]
    vmask = results['pred_vol'] > vthresh
    
    for j in range(len(conf_bins_edges) - 1):
        lo, hi = conf_bins_edges[j], conf_bins_edges[j + 1]
        cmask = (results['confidence'] >= lo) & (results['confidence'] < hi)
        subset = results[vmask & cmask]
        
        if len(subset) > 50:
            correct = (subset['pred_dir'] == subset['actual_dir']).mean()
            heatmap[i, j] = correct
            counts[i, j] = len(subset)
        else:
            heatmap[i, j] = np.nan

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
fig.patch.set_facecolor('#080c14')

for ax in [ax1, ax2]:
    ax.set_facecolor('#080c14')
    ax.tick_params(colors='white')
    for spine in ax.spines.values(): spine.set_edgecolor('#1a2332')

# Accuracy heatmap
conf_labels = [f'{conf_bins_edges[j]:.2f}-{conf_bins_edges[j+1]:.2f}' for j in range(len(conf_bins_edges)-1)]
vol_labels  = [f'Top {100-vp}%' for vp in vol_pcts]

im1 = ax1.imshow(heatmap, cmap='RdYlGn', vmin=0.48, vmax=0.58, aspect='auto')
ax1.set_xticks(range(len(conf_labels))); ax1.set_xticklabels(conf_labels, rotation=45, fontsize=8, color='white')
ax1.set_yticks(range(len(vol_labels))); ax1.set_yticklabels(vol_labels, fontsize=9, color='white')
ax1.set_xlabel('Direction confidence', color='white')
ax1.set_ylabel('Vol filter (training thresholds)', color='white')
ax1.set_title('Accuracy Heatmap', color='white')
for i in range(heatmap.shape[0]):
    for j in range(heatmap.shape[1]):
        if not np.isnan(heatmap[i,j]):
            ax1.text(j, i, f'{heatmap[i,j]:.1%}', ha='center', va='center', fontsize=8, color='black')
plt.colorbar(im1, ax=ax1)

# Count heatmap
im2 = ax2.imshow(counts, cmap='Blues', aspect='auto')
ax2.set_xticks(range(len(conf_labels))); ax2.set_xticklabels(conf_labels, rotation=45, fontsize=8, color='white')
ax2.set_yticks(range(len(vol_labels))); ax2.set_yticklabels(vol_labels, fontsize=9, color='white')
ax2.set_xlabel('Direction confidence', color='white')
ax2.set_ylabel('Vol filter (training thresholds)', color='white')
ax2.set_title('Sample Count', color='white')
for i in range(counts.shape[0]):
    for j in range(counts.shape[1]):
        if counts[i,j] > 0:
            ax2.text(j, i, f'{int(counts[i,j]):,}', ha='center', va='center', fontsize=7, color='white' if counts[i,j] < counts.max()/2 else 'black')
plt.colorbar(im2, ax=ax2)

plt.suptitle('Accuracy × Vol Filter × Confidence — Test Set\n(Thresholds from training set)', color='white', fontsize=13)
plt.tight_layout()
plt.show()

## 9. Summary

In [ ]:
print('=' * 60)
print('TWO-STAGE MODEL â€” TRAINING COMPLETE')
print('=' * 60)

print(f'\nModels saved to: {MODELS_DIR.resolve()}')
for f in sorted(MODELS_DIR.glob('*.joblib')):
    print(f'  {f.name:<25} {f.stat().st_size / 1024 / 1024:.1f} MB')

print(f'\nâ”€â”€ Stage 1: Volatility Model â”€â”€')
print(f'  CV MAE: {np.mean(vol_fold_mae):.6f}')
print(f'  Target: |log_return_1H|')

print(f'\nâ”€â”€ Stage 2: Direction Model â”€â”€')
print(f'  CV Accuracy: {np.mean(dir_fold_acc)*100:.1f}%')
print(f'  Trained on: high-vol rows only (|return| > {vol_threshold:.6f})')

# Best combo from test
best_name, best_acc, best_sharpe = '', 0, 0
for name, thresh in vol_thresholds.items():
    mask = results['pred_vol'] > thresh
    if mask.sum() < 100: continue
    subset = results[mask]
    pnl = subset['pred_dir'] * subset['actual_return']
    acc = (subset['pred_dir'] == subset['actual_dir']).mean()
    sharpe = (pnl.mean() / pnl.std()) * np.sqrt(252 * 24) if pnl.std() > 0 else 0
    if sharpe > best_sharpe:
        best_name, best_acc, best_sharpe = name, acc, sharpe

print(f'\nâ”€â”€ Best test config: {best_name} â”€â”€')
print(f'  Accuracy: {best_acc:.1%}')
print(f'  Sharpe:   {best_sharpe:.2f}')
print(f'\nTest period: {results.index.min().date()} â†’ {results.index.max().date()} (UNSEEN)')
print(f'Training cutoff: {TRAIN_END}')